# 05: Stress Regime (Nov 2025)

Full pipeline repeat on the November 2025 crash month.

**Research question:** Does the functional form of crypto market impact change under stress? Does delta move closer to 0.5? Does PySR extract a formula that looks more like the equity model?

Sections:
1. Data and metaorder reconstruction
2. OLS benchmarks
3. MLP-A and MLP-B
4. Sensitivity analysis and PySR
5. Regime comparison

In [ ]:
import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from pathlib import Path
from scipy import stats
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import joblib
import json
import sys
sys.path.append("../src")

DATA_DIR = Path("../data/processed")
OUTPUT_DIR = Path("../outputs")
MODEL_DIR = OUTPUT_DIR / "models"
FIGURES_DIR = OUTPUT_DIR / "figures"
MODEL_DIR.mkdir(parents=True, exist_ok=True)
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

torch.manual_seed(42)
np.random.seed(42)

plt.rcParams.update({
    "figure.dpi": 120,
    "axes.spines.top": False,
    "axes.spines.right": False,
})


---
## 1. Data and metaorder reconstruction

Nov 2025 data is processed separately from the calm regime. The same pipeline
applies: fetch_data.py (MONTHS=["2025-11"]), process_data.py, reconstruct_metaorders.py.
Output files are named with a _stress suffix to avoid overwriting calm data.

In [ ]:
from reconstruct_metaorders import (
    build_trader_weights,
    assign_trader_ids,
    extract_metaorders,
    compute_daily_stats,
    normalize_metaorders,
)

STRESS_TRADES_PATH = DATA_DIR / "impact_data_stress.parquet"
STRESS_METAORDERS_PATH = DATA_DIR / "metaorders_stress.parquet"

N_TRADERS = 20
DISTRIBUTION = "power_law"
ALPHA = 2.0
MIN_CHILD = 10

if not STRESS_METAORDERS_PATH.exists():
    trades = pd.read_parquet(STRESS_TRADES_PATH)
    daily_stats = compute_daily_stats(trades)
    weights = build_trader_weights(N_TRADERS, DISTRIBUTION, ALPHA)
    trader_ids = assign_trader_ids(len(trades), N_TRADERS, weights)
    mo = extract_metaorders(trades, trader_ids)
    mo = normalize_metaorders(mo, daily_stats)
    mo = mo.dropna(subset=["Q_norm", "I"])
    mo = mo[(mo["sigma_daily"] > 0) & (mo["V_daily"] > 0)]
    mo.to_parquet(STRESS_METAORDERS_PATH, index=False)

mo = pd.read_parquet(STRESS_METAORDERS_PATH)

print(f"Metaorders: {len(mo):,}")
print(f"Median n_child: {mo['n_child'].median():.1f}")
print(f"Median Q_norm: {mo['Q_norm'].median():.2e}")
print(f"sigma_daily mean: {mo['sigma_daily'].mean():.4f}")
print(f"V_daily mean: {mo['V_daily'].mean():,.0f} BTC/day")
mo[["n_child", "Q", "Q_norm", "I"]].describe()


### Log-log plot: metaorder impact vs size

In [ ]:
mo_pos = mo[(mo["I"] > 0) & (mo["Q_norm"] > 0)].copy()

log_q = np.log(mo_pos["Q_norm"].values)
log_i = np.log(mo_pos["I"].values)
mask = (
    (np.abs(log_q - log_q.mean()) < 3 * log_q.std()) &
    (np.abs(log_i - log_i.mean()) < 3 * log_i.std())
)
mo_pos = mo_pos[mask].copy()

N_BINS = 50
mo_pos["q_bin"] = pd.qcut(mo_pos["Q_norm"], q=N_BINS, duplicates="drop")
binned = mo_pos.groupby("q_bin", observed=True).agg(
    Q_median=("Q_norm", "median"),
    I_median=("I", "median"),
    count=("I", "count"),
).reset_index()
binned = binned[binned["count"] >= 50]

bq = np.log(binned["Q_median"].values)
bi = np.log(binned["I_median"].values)

slope, intercept, r, p, se = stats.linregress(bq, bi)
print(f"delta (slope): {slope:.4f}")
print(f"95% CI: [{slope - 1.96*se:.4f}, {slope + 1.96*se:.4f}]")
print(f"R²: {r**2:.4f}")
print(f"p-value: {p:.2e}")


In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))

ax.scatter(bq, bi, s=20, color="steelblue", alpha=0.8, label="Binned median impact")

x_line = np.linspace(bq.min(), bq.max(), 200)
ax.plot(x_line, intercept + slope * x_line,
        color="firebrick", linewidth=1.5,
        label=f"OLS fit  delta = {slope:.3f}")
ax.plot(x_line, intercept + 0.5 * (x_line - bq.mean()) + bi.mean(),
        color="gray", linewidth=1.5, linestyle="--",
        label="Square-root law  delta = 0.5")

ax.set_xlabel("log(Q_norm), metaorder participation rate")
ax.set_ylabel("log(I), normalized impact")
ax.set_title("Metaorder Price Impact vs Size, BTC/USDT (stress regime)")
ax.legend()

plt.tight_layout()
plt.savefig(FIGURES_DIR / "loglog_metaorders_stress.png", bbox_inches="tight")
plt.show()

### Sigma comparison: calm vs stress

In [ ]:
mo_calm = pd.read_parquet(DATA_DIR / "metaorders.parquet")

print(f"sigma_daily mean (calm): {mo_calm['sigma_daily'].mean():.4f}")
print(f"sigma_daily mean (stress): {mo['sigma_daily'].mean():.4f}")
print(f"Ratio: {mo['sigma_daily'].mean() / mo_calm['sigma_daily'].mean():.1f}x")


### Train/test split

In [ ]:
mo = mo.sort_values("start_ts").reset_index(drop=True)
split_idx = int(len(mo) * 0.8)

mo_train = mo.iloc[:split_idx]
mo_test = mo.iloc[split_idx:]

mo_train.to_parquet(DATA_DIR / "mo_train_stress.parquet", index=False)
mo_test.to_parquet(DATA_DIR / "mo_test_stress.parquet", index=False)

t_min_train = pd.to_datetime(mo_train["start_ts"].min(), unit="ms").date()
t_max_train = pd.to_datetime(mo_train["start_ts"].max(), unit="ms").date()
t_min_test = pd.to_datetime(mo_test["start_ts"].min(), unit="ms").date()
t_max_test = pd.to_datetime(mo_test["start_ts"].max(), unit="ms").date()

print(f"Train: {len(mo_train):,} metaorders  ({t_min_train} to {t_max_train})")
print(f"Test: {len(mo_test):,} metaorders  ({t_min_test} to {t_max_test})")


---
## 2. OLS benchmarks

Same benchmarks as 02_benchmark.ipynb, fit on stress training data.

In [ ]:
def prepare(df: pd.DataFrame) -> pd.DataFrame:
    df = df[(df["I"] > 0) & (df["Q_norm"] > 0) & (df["sigma_daily"] > 0)].copy()
    df["log_I"] = np.log(df["I"])
    df["log_Q"] = np.log(df["Q_norm"])
    df["log_sigma"] = np.log(df["sigma_daily"])
    df["log_V"] = np.log(df["V_daily"])
    df["log_n_child"] = np.log(df["n_child"])
    hour = pd.to_datetime(df["start_ts"], unit="ms").dt.hour
    df["utc_hour_sin"] = np.sin(2 * np.pi * hour / 24)
    df["utc_hour_cos"] = np.cos(2 * np.pi * hour / 24)
    mu, sd = df["log_I"].mean(), df["log_I"].std()
    df = df[np.abs(df["log_I"] - mu) < 3 * sd].reset_index(drop=True)
    return df

train_p = prepare(mo_train)
test_p = prepare(mo_test)

print(f"Train after prep: {len(train_p):,}")
print(f"Test after prep: {len(test_p):,}")
print()
print("log_I distribution (train):")
print(train_p["log_I"].describe().round(3))
print()
print("log_sigma distribution (train):")
print(train_p["log_sigma"].describe().round(3))


In [ ]:
y_train = train_p["log_I"].values
y_test = test_p["log_I"].values

# Power law
X_train_b1 = train_p[["log_Q"]].values
X_test_b1 = test_p[["log_Q"]].values
b1 = LinearRegression().fit(X_train_b1, y_train)
slope_b1, _, _, _, se_b1 = stats.linregress(train_p["log_Q"].values, y_train)

print("Power law:")
print(f"  delta: {b1.coef_[0]:.4f}")
print(f"  95% CI: [{slope_b1 - 1.96*se_b1:.4f}, {slope_b1 + 1.96*se_b1:.4f}]")
print(f"  Train R2: {r2_score(y_train, b1.predict(X_train_b1)):.4f}")
print()

# Almgren-Chriss
X_train_b2 = train_p[["log_Q", "log_sigma"]].values
X_test_b2 = test_p[["log_Q", "log_sigma"]].values
b2 = LinearRegression().fit(X_train_b2, y_train)

print("Almgren-Chriss:")
print(f"  delta: {b2.coef_[0]:.4f}  (expected 0.5)")
print(f"  beta: {b2.coef_[1]:.4f}  (expected 1.0)")
print(f"  Train R2: {r2_score(y_train, b2.predict(X_train_b2)):.4f}")

# Constrained delta = 0.5
intercept_constrained = (y_train - 0.5 * train_p["log_Q"].values).mean()
y_pred_constrained = intercept_constrained + 0.5 * test_p["log_Q"].values


In [ ]:
def evaluate(name: str, y_true: np.ndarray, y_pred: np.ndarray) -> dict:
    mse = mean_squared_error(y_true, y_pred)
    mae = mean_absolute_error(y_true, y_pred)
    r2 = r2_score(y_true, y_pred)
    bias = (y_pred - y_true).mean()
    print(f"  {name:<35} MSE={mse:.4f}  MAE={mae:.4f}  R2={r2:.4f}  Bias={bias:.4f}")
    return {"model": name, "mse": mse, "mae": mae, "r2": r2, "bias": bias}

print("Out-of-sample results (stress):")
benchmark_results = [
    evaluate("Power law (delta=0.5, constrained)", y_test, y_pred_constrained),
    evaluate("Power law (delta fitted)",           y_test, b1.predict(X_test_b1)),
    evaluate("Almgren-Chriss",                     y_test, b2.predict(X_test_b2)),
]

pd.DataFrame(benchmark_results).to_csv(OUTPUT_DIR / "benchmark_results_stress.csv", index=False)


---
## 3. MLP-A and MLP-B

In [ ]:
class ImpactMLP(nn.Module):
    def __init__(self, input_dim: int, hidden: int = 64, layers: int = 3, dropout: float = 0.3):
        super().__init__()
        net = [nn.Linear(input_dim, hidden), nn.ReLU(), nn.Dropout(dropout)]
        for _ in range(layers - 1):
            net += [nn.Linear(hidden, hidden), nn.ReLU(), nn.Dropout(dropout)]
        net += [nn.Linear(hidden, 1)]
        self.net = nn.Sequential(*net)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.net(x).squeeze(-1)


def train_mlp(
    X_train: np.ndarray,
    y_train: np.ndarray,
    X_val: np.ndarray,
    y_val: np.ndarray,
    input_dim: int,
    hidden: int = 64,
    layers: int = 3,
    epochs: int = 100,
    batch_size: int = 512,
    lr: float = 1e-3,
    patience: int = 10,
) -> tuple[ImpactMLP, list, list]:

    model = ImpactMLP(input_dim, hidden, layers).to(DEVICE)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)
    criterion = nn.MSELoss()

    X_tr = torch.tensor(X_train, dtype=torch.float32)
    y_tr = torch.tensor(y_train, dtype=torch.float32)
    X_vl = torch.tensor(X_val, dtype=torch.float32).to(DEVICE)
    y_vl = torch.tensor(y_val, dtype=torch.float32).to(DEVICE)

    loader = DataLoader(
        TensorDataset(X_tr, y_tr),
        batch_size=batch_size,
        shuffle=True,
    )

    train_losses, val_losses = [], []
    best_val_loss = float("inf")
    best_state = None
    patience_ctr = 0

    for epoch in range(epochs):
        model.train()
        epoch_loss = 0.0
        for Xb, yb in loader:
            Xb, yb = Xb.to(DEVICE), yb.to(DEVICE)
            optimizer.zero_grad()
            loss = criterion(model(Xb), yb)
            loss.backward()
            optimizer.step()
            epoch_loss += loss.item() * len(Xb)
        scheduler.step()

        model.eval()
        with torch.no_grad():
            val_loss = criterion(model(X_vl), y_vl).item()

        train_losses.append(epoch_loss / len(X_train))
        val_losses.append(val_loss)

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_state = {k: v.clone() for k, v in model.state_dict().items()}
            patience_ctr = 0
        else:
            patience_ctr += 1
            if patience_ctr >= patience:
                print(f"  Early stopping at epoch {epoch + 1}")
                break

        if (epoch + 1) % 10 == 0:
            print(f"  Epoch {epoch+1:3d}  train={train_losses[-1]:.4f}  val={val_loss:.4f}")

    model.load_state_dict(best_state)
    return model, train_losses, val_losses


def predict(model: ImpactMLP, X: np.ndarray) -> np.ndarray:
    model.eval()
    with torch.no_grad():
        t = torch.tensor(X, dtype=torch.float32).to(DEVICE)
        return model(t).cpu().numpy()


In [ ]:
FEATURES_A = ["log_Q", "log_sigma", "log_V"]
FEATURES_B = ["log_Q", "log_sigma", "log_V", "log_n_child", "utc_hour_sin", "utc_hour_cos"]

val_cut = int(len(train_p) * 0.9)
tr = train_p.iloc[:val_cut]
vl = train_p.iloc[val_cut:]

y_te = test_p["log_I"].values

# MLP-A
scaler_a = StandardScaler().fit(tr[FEATURES_A].values)
joblib.dump(scaler_a, MODEL_DIR / "scaler_a_stress.pkl")

mlp_a, train_loss_a, val_loss_a = train_mlp(
    scaler_a.transform(tr[FEATURES_A].values), tr["log_I"].values,
    scaler_a.transform(vl[FEATURES_A].values), vl["log_I"].values,
    input_dim=len(FEATURES_A),
)
torch.save(mlp_a.state_dict(), MODEL_DIR / "mlp_a_stress.pt")

# MLP-B
scaler_b = StandardScaler().fit(tr[FEATURES_B].values)
joblib.dump(scaler_b, MODEL_DIR / "scaler_b_stress.pkl")

mlp_b, train_loss_b, val_loss_b = train_mlp(
    scaler_b.transform(tr[FEATURES_B].values), tr["log_I"].values,
    scaler_b.transform(vl[FEATURES_B].values), vl["log_I"].values,
    input_dim=len(FEATURES_B),
)
torch.save(mlp_b.state_dict(), MODEL_DIR / "mlp_b_stress.pt")


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(train_loss_a, label="Train", color="steelblue")
axes[0].plot(val_loss_a,   label="Val",   color="firebrick")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("MSE")
axes[0].set_title("MLP-A training curve (stress)")
axes[0].legend()

axes[1].plot(train_loss_b, label="Train", color="steelblue")
axes[1].plot(val_loss_b,   label="Val",   color="firebrick")
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("MSE")
axes[1].set_title("MLP-B training curve (stress)")
axes[1].legend()

plt.tight_layout()
plt.savefig(FIGURES_DIR / "mlp_training_curves_stress.png", bbox_inches="tight")
plt.show()

In [ ]:
X_te_a = scaler_a.transform(test_p[FEATURES_A].values)
X_te_b = scaler_b.transform(test_p[FEATURES_B].values)

pred_a = predict(mlp_a, X_te_a)
pred_b = predict(mlp_b, X_te_b)

print("Out-of-sample results (stress):")
mlp_results = [
    evaluate("MLP-A (log_Q, log_sigma, log_V)", y_te, pred_a),
    evaluate("MLP-B (+ n_child, utc_hour)",     y_te, pred_b),
]

pd.DataFrame(mlp_results).to_csv(OUTPUT_DIR / "mlp_results_stress.csv", index=False)

---
## 4. Sensitivity analysis and PySR

Same procedure as 04_interpretability.ipynb. MLP-A only, inputs match OLS benchmarks exactly.

In [ ]:
medians = train_p[FEATURES_A].median().values
p01 = train_p[FEATURES_A].quantile(0.01).values
p99 = train_p[FEATURES_A].quantile(0.99).values

N_SWEEP = 200
sensitivity = {}

for i, feature in enumerate(FEATURES_A):
    sweep = np.linspace(p01[i], p99[i], N_SWEEP)
    X_sweep = np.tile(medians, (N_SWEEP, 1))
    X_sweep[:, i] = sweep
    y_pred = predict(mlp_a, scaler_a.transform(X_sweep))
    sensitivity[feature] = {"sweep": sweep, "pred": y_pred}
    print(f"{feature}: range [{p01[i]:.3f}, {p99[i]:.3f}]  pred range [{y_pred.min():.3f}, {y_pred.max():.3f}]")


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

labels = {
    "log_Q":     "log(Q_norm), participation rate",
    "log_sigma": "log(sigma_daily), daily realized vol",
    "log_V":     "log(V_daily), daily volume",
}

for ax, feature in zip(axes, FEATURES_A):
    s = sensitivity[feature]
    ax.plot(s["sweep"], s["pred"], color="steelblue", linewidth=2)
    ax.axvline(medians[FEATURES_A.index(feature)], color="gray",
               linestyle="--", linewidth=1, label="Median")
    ax.set_xlabel(labels[feature])
    ax.set_ylabel("Predicted log I")
    ax.set_title(f"Sensitivity: {feature}")
    ax.legend()

plt.suptitle("MLP-A sensitivity analysis, stress regime", y=1.02)
plt.tight_layout()
plt.savefig(FIGURES_DIR / "sensitivity_stress.png", bbox_inches="tight")
plt.show()

In [ ]:
s = sensitivity["log_Q"]
dlog_I_dlog_Q = np.gradient(s["pred"], s["sweep"])

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(s["sweep"], s["pred"], color="steelblue", linewidth=2)
axes[0].set_xlabel("log(Q_norm)")
axes[0].set_ylabel("Predicted log I")
axes[0].set_title("log_Q sensitivity curve (stress)")

calm_summary = json.load(open(OUTPUT_DIR / "calm_summary.json"))
CALM_OLS_DELTA = calm_summary["ols_delta"]
axes[1].plot(s["sweep"], dlog_I_dlog_Q, color="firebrick", linewidth=2,
             label="d(log I)/d(log Q)")
axes[1].axhline(0.5, color="gray", linestyle="--", linewidth=1.5, label="Sqrt law (0.5)")
axes[1].axhline(CALM_OLS_DELTA, color="steelblue", linestyle=":", linewidth=1.5, label=f"Calm OLS delta ({CALM_OLS_DELTA})")
axes[1].axhline(0.0, color="black", linestyle="-", linewidth=0.8, alpha=0.4)
axes[1].set_xlabel("log(Q_norm)")
axes[1].set_ylabel("Local slope")
axes[1].set_title("Local power-law exponent, MLP-A (stress)")
axes[1].legend()

plt.tight_layout()
plt.savefig(FIGURES_DIR / "local_slope_stress.png", bbox_inches="tight")
plt.show()

print(f"Mean local slope: {dlog_I_dlog_Q.mean():.4f}")
print(f"Slope range: [{dlog_I_dlog_Q.min():.4f}, {dlog_I_dlog_Q.max():.4f}]")
print(f"Slope std: {dlog_I_dlog_Q.std():.4f}")


### PySR symbolic regression

In [ ]:
from pysr import PySRRegressor

N_PYSR = 40_000
N_BINS_PYSR = 20
PER_BIN = N_PYSR // N_BINS_PYSR

train_pysr = train_p.copy()
train_pysr["q_bin"] = pd.qcut(train_pysr["log_Q"], q=N_BINS_PYSR, duplicates="drop")

frames = []
for _, grp in train_pysr.groupby("q_bin", observed=True):
    n = min(PER_BIN, len(grp))
    frames.append(grp.sample(n=n, random_state=42))

sample = pd.concat(frames).reset_index(drop=True)
print(f"PySR sample size: {len(sample):,}")

X_pysr_raw = sample[FEATURES_A].values
X_pysr = scaler_a.transform(X_pysr_raw)
y_pysr = predict(mlp_a, X_pysr)


In [ ]:
model_pysr = PySRRegressor(
    niterations=50,
    binary_operators=["+", "-", "*", "/", "^"],
    unary_operators=["log", "exp", "sqrt", "abs"],
    constraints={"^": (-1, 1)},
    maxsize=20,
    populations=30,
    population_size=50,
    parsimony=0.01,
    random_state=42,
    verbosity=1,
    temp_equation_file=True,
)

model_pysr.fit(X_pysr_raw, y_pysr, variable_names=FEATURES_A)
print(model_pysr)
print(model_pysr.get_best())


In [ ]:
equations = model_pysr.equations_.copy()

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(equations["complexity"], equations["loss"],
        marker="o", color="steelblue", linewidth=1.5)
for _, row in equations.iterrows():
    label = str(row["equation"])[:30]
    ax.annotate(label, (row["complexity"], row["loss"]),
                textcoords="offset points", xytext=(5, 0),
                fontsize=6, alpha=0.7)
ax.set_xlabel("Complexity")
ax.set_ylabel("Loss (MSE on PySR sample)")
ax.set_title("PySR Pareto front, stress regime")
ax.set_yscale("log")
plt.tight_layout()
plt.savefig(FIGURES_DIR / "pysr_pareto_stress.png", bbox_inches="tight")
plt.show()

In [ ]:
MIN_COMPLEXITY = 5

equations = equations.sort_values("complexity").reset_index(drop=True)
candidates = equations[equations["complexity"] >= MIN_COMPLEXITY].reset_index(drop=True)

if len(candidates) < 2:
    elbow_idx_in_candidates = 0
else:
    loss_drop = candidates["loss"].values[:-1] - candidates["loss"].values[1:]
    efficiency = loss_drop / np.ones(len(loss_drop))
    elbow_idx_in_candidates = int(np.argmax(efficiency)) + 1

best_eq = candidates.iloc[elbow_idx_in_candidates]
elbow_idx = best_eq.name

print(f"Selected (elbow, complexity >= {MIN_COMPLEXITY}):")
print(f"  Complexity: {best_eq['complexity']}")
print(f"  Loss: {best_eq['loss']:.6f}")
print(f"  Equation: {best_eq['equation']}")


In [ ]:
X_te_raw = test_p[FEATURES_A].values
X_te = scaler_a.transform(X_te_raw)
y_te_mlp = predict(mlp_a, X_te)
y_te_true = test_p["log_I"].values
y_te_pysr = model_pysr.predict(X_te_raw, index=elbow_idx)

print("PySR vs MLP predictions (stress test set):")
print(f"  MSE: {mean_squared_error(y_te_mlp, y_te_pysr):.4f}")
print(f"  R2: {r2_score(y_te_mlp, y_te_pysr):.4f}")
print()
print("PySR vs actual log_I:")
print(f"  MSE: {mean_squared_error(y_te_true, y_te_pysr):.4f}")
print(f"  R2: {r2_score(y_te_true, y_te_pysr):.4f}")
print()
print("MLP-A vs actual log_I:")
print(f"  MSE: {mean_squared_error(y_te_true, y_te_mlp):.4f}")
print(f"  R2: {r2_score(y_te_true, y_te_mlp):.4f}")


---
## 5. Regime comparison

Calm results are loaded from outputs saved in notebooks 02-04.

In [ ]:
calm_benchmarks = pd.read_csv(OUTPUT_DIR / "benchmark_results.csv")
calm_mlp = pd.read_csv(OUTPUT_DIR / "mlp_results.csv")

stress_benchmarks = pd.read_csv(OUTPUT_DIR / "benchmark_results_stress.csv")
stress_mlp = pd.read_csv(OUTPUT_DIR / "mlp_results_stress.csv")

print("OOS MSE, calm vs stress:")
print(f"{'Model':<35} {'Calm MSE':>10} {'Stress MSE':>12}")
print("-" * 60)
for (_, c), (_, s) in zip(calm_benchmarks.iterrows(), stress_benchmarks.iterrows()):
    print(f"  {c['model']:<33} {c['mse']:>10.4f} {s['mse']:>12.4f}")
for (_, c), (_, s) in zip(calm_mlp.iterrows(), stress_mlp.iterrows()):
    print(f"  {c['model']:<33} {c['mse']:>10.4f} {s['mse']:>12.4f}")


In [ ]:
# Sweep log_Q with sigma and V at their respective regime medians
median_sigma_stress = train_p["log_sigma"].median()
median_V_stress = train_p["log_V"].median()

mo_train_calm = pd.read_parquet(DATA_DIR / "mo_train.parquet")
train_p_calm = prepare(mo_train_calm)
median_sigma_calm = train_p_calm["log_sigma"].median()
median_V_calm = train_p_calm["log_V"].median()

q_range = np.linspace(
    train_p["log_Q"].quantile(0.01),
    train_p["log_Q"].quantile(0.99),
    200,
)

X_stress_raw = np.column_stack([
    q_range,
    np.full_like(q_range, median_sigma_stress),
    np.full_like(q_range, median_V_stress),
])

X_calm_raw = np.column_stack([
    q_range,
    np.full_like(q_range, median_sigma_calm),
    np.full_like(q_range, median_V_calm),
])

# Load calm MLP-A
mlp_a_calm = ImpactMLP(input_dim=len(FEATURES_A)).to(DEVICE)
mlp_a_calm.load_state_dict(torch.load(MODEL_DIR / "mlp_a.pt", map_location=DEVICE))
scaler_a_calm = joblib.load(MODEL_DIR / "scaler_a.pkl")

y_stress_mlp = predict(mlp_a, scaler_a.transform(X_stress_raw))
y_stress_pysr = model_pysr.predict(X_stress_raw, index=elbow_idx)
y_calm_mlp = predict(mlp_a_calm, scaler_a_calm.transform(X_calm_raw))

# Load calm PySR model
model_pysr_calm = joblib.load(MODEL_DIR / "pysr_calm.pkl")
calm_elbow_idx = joblib.load(MODEL_DIR / "pysr_calm_elbow_idx.pkl")
y_calm_pysr = model_pysr_calm.predict(X_calm_raw, index=calm_elbow_idx)

# Sqrt law reference slope
mid = len(q_range) // 2
y_sqrt_line = 0.5 * (q_range - q_range[mid])

# Normalize to zero at left edge to compare slopes only
y_calm_mlp_norm = y_calm_mlp - y_calm_mlp[0]
y_calm_pysr_norm = y_calm_pysr - y_calm_pysr[0]
y_stress_mlp_norm = y_stress_mlp - y_stress_mlp[0]
y_stress_pysr_norm = y_stress_pysr - y_stress_pysr[0]
y_sqrt_norm = y_sqrt_line - y_sqrt_line[0]

fig, ax = plt.subplots(figsize=(9, 5))

ax.plot(q_range, y_calm_mlp_norm, color="steelblue", linewidth=2, linestyle="--", label="MLP-A (calm)")
ax.plot(q_range, y_calm_pysr_norm, color="steelblue", linewidth=1.5, linestyle=":", label="PySR (calm)")
ax.plot(q_range, y_stress_mlp_norm, color="firebrick", linewidth=2, linestyle="--", label="MLP-A (stress)")
ax.plot(q_range, y_stress_pysr_norm, color="firebrick", linewidth=1.5, linestyle=":", label="PySR (stress)")
ax.plot(q_range, y_sqrt_norm, color="gray", linewidth=1.5, linestyle="-", label="Sqrt law (delta=0.5)")

ax.set_xlabel("log(Q_norm)")
ax.set_ylabel("Predicted log I (normalized to zero)")
ax.set_title("Impact formula slope comparison: calm vs stress")
ax.legend()

plt.tight_layout()
plt.savefig(FIGURES_DIR / "formula_comparison_regimes.png", bbox_inches="tight")
plt.show()


---
## Summary

In [ ]:
stress_delta = b1.coef_[0]
stress_mlp_slope = dlog_I_dlog_Q.mean()

stress_summary = {
    "ols_delta": float(stress_delta),
    "mlp_mean_slope": float(stress_mlp_slope),
    "pysr_complexity": int(best_eq["complexity"]),
    "pysr_loss": float(best_eq["loss"]),
    "pysr_equation": str(best_eq["equation"]),
}
pd.Series(stress_summary).to_json(OUTPUT_DIR / "stress_summary.json")

joblib.dump(model_pysr, MODEL_DIR / "pysr_stress.pkl")
joblib.dump(elbow_idx, MODEL_DIR / "pysr_stress_elbow_idx.pkl")

calm_summary = json.load(open(OUTPUT_DIR / "calm_summary.json"))
calm_ols_delta = calm_summary["ols_delta"]
calm_mlp_slope = calm_summary["mlp_mean_slope"]
calm_pysr_eq = calm_summary["pysr_equation"]

print("Regime comparison:")
print(f"  {'':30} {'Calm':>10} {'Stress':>10}")
print(f"  {'-'*52}")
print(f"  {'OLS delta':<30} {calm_ols_delta:>10.3f} {stress_delta:>10.3f}")
print(f"  {'MLP-A mean local slope':<30} {calm_mlp_slope:>10.3f} {stress_mlp_slope:>10.3f}")
print(f"  {'PySR formula':<30} {calm_pysr_eq[:16]:>16} {str(best_eq['equation'])[:10]:>10}")
print(f"  {'Matches equity model?':<30} {'No':>10} {'No':>10}")
